# 00 · 실험 러너

**이 노트북은 고치지 않는다.** 아래 두 줄만 바꾸고 전부 실행하면 된다.

실험 코드는 `experiments/local/` 에 파이썬 파일로 둔다. 작성법은
[`experiments/local/README.md`](https://github.com/hyunku9566/lga_data/blob/main/experiments/local/README.md) 참고.

```
로컬에서 파일 작성 → dryrun 검증 → git push → 여기서 실행 → 원장에 결과
```


In [ ]:
RUNNER_NAME = "본인이름"                              # ← 바꾼다
SCRIPT      = "experiments/local/_template.py"        # ← 돌릴 파일
BRANCH      = "main"                                  # 다른 브랜치면 바꾼다


In [ ]:
# ── 준비 (레포 최신화 + 데이터 확보) ────────────────────────────
import os, sys, subprocess
REPO = '/content/lga-repo'
if not os.path.exists(f'{REPO}/src/config.py'):
    subprocess.run(['git','clone','-q','https://github.com/hyunku9566/lga_data.git',REPO])
subprocess.run(['git','-C',REPO,'fetch','-q','origin'])
subprocess.run(['git','-C',REPO,'checkout','-q',BRANCH])
subprocess.run(['git','-C',REPO,'reset','--hard','-q',f'origin/{BRANCH}'])
subprocess.run(['pip','install','-q','-r',f'{REPO}/requirements-colab.txt'])

for m in [k for k in list(sys.modules)
          if k.startswith('src') or k in ('config','lib_lga','experiment','download','bootstrap','dryrun')]:
    del sys.modules[m]
for _p in (f'{REPO}/src', REPO):
    if _p not in sys.path: sys.path.insert(0, _p)

if not os.environ.get('LGA_PASSWORD'):
    from getpass import getpass
    os.environ['LGA_PASSWORD'] = getpass('team 비밀번호 (Drive 에 캐시 있으면 엔터): ').strip()

from bootstrap import setup
C = setup(runner=RUNNER_NAME)
print(f'\n현재 커밋: ' + subprocess.run(['git','-C',REPO,'log','--oneline','-1'],
                                        capture_output=True,text=True).stdout.strip())


## 실행 전 검증

데이터를 쓰지 않고 인자·격자·예상 소요시간만 확인한다. 여기서 걸리면 파일을 고쳐서 다시 push 해라.

In [ ]:
import dryrun
r = dryrun.dry_run(f'{REPO}/{SCRIPT}')
assert not r['problems'], '문제를 고치고 다시 push 해라'


## 실행

오래 걸린다. 조합마다 결과가 **즉시 원장에 기록**되므로 중간에 런타임이 끊겨도 이전 것은 남는다.

In [ ]:
import runpy, time
t0 = time.time()
os.environ['LGA_RUNNER'] = RUNNER_NAME
runpy.run_path(f'{REPO}/{SCRIPT}', run_name='__main__')
print(f'\n총 {(time.time()-t0)/60:.1f}분')


## 내 결과만 확인

In [ ]:
import experiment as E
led = E.read_all_ledgers()
mine = led[led.runner == RUNNER_NAME] if len(led) else led
print(f'{RUNNER_NAME} 의 기록 {len(mine)}건')
if len(mine):
    display(mine[['name','m24','delta24','m23','delta23','verdict','seeds']].tail(20))
    print()
    print(mine.verdict.value_counts().to_string())
